In [1]:
from google.colab import files
uploaded = files.upload()

Saving Sample - Superstore.csv to Sample - Superstore.csv


In [2]:
import pandas as pd

df = pd.read_csv('Sample - Superstore.csv', encoding='latin1')

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace('-', '_')
)

print(df.columns.tolist())
df.head()

['row_id', 'order_id', 'order_date', 'ship_date', 'ship_mode', 'customer_id', 'customer_name', 'segment', 'country', 'city', 'state', 'postal_code', 'region', 'product_id', 'category', 'sub_category', 'product_name', 'sales', 'quantity', 'discount', 'profit']


,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [3]:
import sqlite3
conn = sqlite3.connect('superstore.db')
cursor = conn.cursor()

In [4]:
df.to_sql('superstore_raw', conn, if_exists='replace', index=False)
pd.read_sql_query("SELECT COUNT(*) FROM superstore_raw;", conn)

,COUNT(*)
0,9994


In [5]:
cursor.executescript("""
DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS products;
DROP TABLE IF EXISTS customers;

CREATE TABLE customers (
    customer_id     TEXT PRIMARY KEY,
    customer_name   TEXT,
    segment         TEXT,
    country         TEXT,
    city            TEXT,
    state           TEXT,
    postal_code     TEXT,
    region          TEXT
);

CREATE TABLE products (
    product_id      TEXT PRIMARY KEY,
    category        TEXT,
    sub_category    TEXT,
    product_name    TEXT
);

CREATE TABLE orders (
    row_id          INTEGER PRIMARY KEY,
    order_id        TEXT,
    customer_id     TEXT,
    product_id      TEXT,
    order_date      TEXT,
    ship_date       TEXT,
    ship_mode       TEXT,
    sales           REAL,
    quantity        INTEGER,
    discount        REAL,
    profit          REAL,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id)
);
""")
conn.commit()

In [6]:
cursor.executescript("""
INSERT INTO customers (customer_id, customer_name, segment, country, city, state, postal_code, region)
SELECT customer_id, customer_name, segment, country, city, state, postal_code, region
FROM superstore_raw
GROUP BY customer_id;
""")
conn.commit()
pd.read_sql_query("SELECT COUNT(*) FROM customers;", conn)

,COUNT(*)
0,793


In [7]:
cursor.executescript("""
INSERT INTO products (product_id, category, sub_category, product_name)
SELECT product_id, category, sub_category, product_name
FROM superstore_raw
GROUP BY product_id;
""")
conn.commit()
pd.read_sql_query("SELECT COUNT(*) FROM products;", conn)

,COUNT(*)
0,1862


In [8]:
cursor.executescript("""
INSERT INTO orders (row_id, order_id, customer_id, product_id, order_date, ship_date, ship_mode, sales, quantity, discount, profit)
SELECT row_id, order_id, customer_id, product_id, order_date, ship_date, ship_mode, sales, quantity, discount, profit
FROM superstore_raw;
""")
conn.commit()
pd.read_sql_query("SELECT COUNT(*) FROM orders;", conn)

,COUNT(*)
0,9994


In [9]:
pd.read_sql_query("""
SELECT
    (SELECT COUNT(*) FROM customers) AS customers,
    (SELECT COUNT(*) FROM products) AS products,
    (SELECT COUNT(*) FROM orders) AS orders,
    (SELECT COUNT(*) FROM superstore_raw) AS raw_rows;
""", conn)

,customers,products,orders,raw_rows
0,793,1862,9994,9994


In [10]:
query = """
SELECT order_id, customer_id, product_id, sales
FROM orders
WHERE sales > (SELECT AVG(sales) FROM orders)
ORDER BY sales DESC;
"""
pd.read_sql_query(query, conn)

,order_id,customer_id,product_id,sales
0,CA-2014-145317,SM-20320,TEC-MA-10002412,22638.480
1,CA-2016-118689,TC-20980,TEC-CO-10004722,17499.950
2,CA-2017-140151,RB-19360,TEC-CO-10004722,13999.960
3,CA-2017-127180,TA-21385,TEC-CO-10004722,11199.968
4,CA-2017-166709,HL-15040,TEC-CO-10004722,10499.970
...,...,...,...,...
2355,CA-2014-147298,AG-10300,FUR-CH-10004886,230.280
2356,CA-2017-161956,DR-12880,FUR-CH-10004886,230.280
2357,US-2014-106334,JF-15490,FUR-CH-10004886,230.280
2358,US-2016-168095,MC-17425,FUR-CH-10004886,230.280


In [11]:
query = """
SELECT o.customer_id, o.order_id, o.sales
FROM orders o
WHERE o.sales = (
    SELECT MAX(o2.sales)
    FROM orders o2
    WHERE o2.customer_id = o.customer_id
)
ORDER BY o.customer_id;
"""
pd.read_sql_query(query, conn)

,customer_id,order_id,sales
0,AA-10315,CA-2016-103982,3930.072
1,AA-10375,CA-2016-131065,499.980
2,AA-10480,CA-2016-114601,479.970
3,AA-10645,CA-2015-110863,1323.900
4,AB-10015,CA-2016-140935,341.960
...,...,...,...
790,XP-21865,CA-2014-114335,337.088
791,YC-21895,CA-2014-119375,2934.330
792,YS-21880,CA-2017-119809,2793.528
793,ZC-21910,CA-2015-115567,1516.200


In [12]:
query = """
WITH customer_totals AS (
    SELECT customer_id, SUM(sales) AS total_sales
    FROM orders
    GROUP BY customer_id
)
SELECT c.customer_name, ct.total_sales
FROM customer_totals ct
JOIN customers c ON c.customer_id = ct.customer_id
ORDER BY ct.total_sales DESC;
"""
pd.read_sql_query(query, conn)

,customer_name,total_sales
0,Sean Miller,25043.050
1,Tamara Chand,19052.218
2,Raymond Buch,15117.339
3,Tom Ashbrook,14595.620
4,Adrian Barton,14473.571
...,...,...
788,Roy Skaria,22.328
789,Mitch Gastineau,16.739
790,Carl Jackson,16.520
791,Lela Donovan,5.304


In [13]:
query = """
WITH customer_totals AS (
    SELECT customer_id, SUM(sales) AS total_sales
    FROM orders
    GROUP BY customer_id
)
SELECT c.customer_name, ct.total_sales
FROM customer_totals ct
JOIN customers c ON c.customer_id = ct.customer_id
WHERE ct.total_sales > (SELECT AVG(total_sales) FROM customer_totals)
ORDER BY ct.total_sales DESC;
"""
pd.read_sql_query(query, conn)

,customer_name,total_sales
0,Sean Miller,25043.050
1,Tamara Chand,19052.218
2,Raymond Buch,15117.339
3,Tom Ashbrook,14595.620
4,Adrian Barton,14473.571
...,...,...
289,Julie Kriz,2932.484
290,Shaun Weien,2921.544
291,Maris LaWare,2921.500
292,Rob Dowd,2912.894


In [14]:
query = """
WITH customer_totals AS (
    SELECT customer_id, SUM(sales) AS total_sales
    FROM orders
    GROUP BY customer_id
)
SELECT
    c.customer_name,
    ct.total_sales,
    RANK() OVER (ORDER BY ct.total_sales DESC) AS sales_rank
FROM customer_totals ct
JOIN customers c ON c.customer_id = ct.customer_id
ORDER BY sales_rank;
"""
pd.read_sql_query(query, conn)

,customer_name,total_sales,sales_rank
0,Sean Miller,25043.050,1
1,Tamara Chand,19052.218,2
2,Raymond Buch,15117.339,3
3,Tom Ashbrook,14595.620,4
4,Adrian Barton,14473.571,5
...,...,...,...
788,Roy Skaria,22.328,789
789,Mitch Gastineau,16.739,790
790,Carl Jackson,16.520,791
791,Lela Donovan,5.304,792


In [15]:
query = """
SELECT
    customer_id,
    order_id,
    order_date,
    sales,
    ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date) AS order_seq_num
FROM orders
ORDER BY customer_id, order_seq_num;
"""
pd.read_sql_query(query, conn)

,customer_id,order_id,order_date,sales,order_seq_num
0,AA-10315,CA-2015-121391,10/4/2015,26.960,1
1,AA-10315,CA-2016-103982,3/3/2016,3930.072,2
2,AA-10315,CA-2016-103982,3/3/2016,2.304,3
3,AA-10315,CA-2016-103982,3/3/2016,431.976,4
4,AA-10315,CA-2016-103982,3/3/2016,41.720,5
...,...,...,...,...,...
9989,ZD-21925,CA-2016-152471,7/8/2016,823.960,5
9990,ZD-21925,CA-2016-152471,7/8/2016,15.984,6
9991,ZD-21925,CA-2014-143336,8/27/2014,8.560,7
9992,ZD-21925,CA-2014-143336,8/27/2014,213.480,8


In [16]:
query = """
WITH customer_totals AS (
    SELECT customer_id, SUM(sales) AS total_sales
    FROM orders
    GROUP BY customer_id
),
ranked AS (
    SELECT
        c.customer_name,
        ct.total_sales,
        DENSE_RANK() OVER (ORDER BY ct.total_sales DESC) AS sales_rank
    FROM customer_totals ct
    JOIN customers c ON c.customer_id = ct.customer_id
)
SELECT customer_name, total_sales, sales_rank
FROM ranked
WHERE sales_rank <= 3
ORDER BY sales_rank;
"""
pd.read_sql_query(query, conn)

,customer_name,total_sales,sales_rank
0,Sean Miller,25043.050,1
1,Tamara Chand,19052.218,2
2,Raymond Buch,15117.339,3


In [17]:
query = """
WITH customer_sales AS (
    SELECT o.customer_id, SUM(o.sales) AS total_sales
    FROM orders o
    GROUP BY o.customer_id
)
SELECT
    c.customer_name,
    cs.total_sales,
    RANK() OVER (ORDER BY cs.total_sales DESC) AS rank
FROM customer_sales cs
JOIN customers c ON c.customer_id = cs.customer_id
ORDER BY rank;
"""
pd.read_sql_query(query, conn)

,customer_name,total_sales,rank
0,Sean Miller,25043.050,1
1,Tamara Chand,19052.218,2
2,Raymond Buch,15117.339,3
3,Tom Ashbrook,14595.620,4
4,Adrian Barton,14473.571,5
...,...,...,...
788,Roy Skaria,22.328,789
789,Mitch Gastineau,16.739,790
790,Carl Jackson,16.520,791
791,Lela Donovan,5.304,792


In [18]:
query = """
WITH customer_totals AS (
    SELECT customer_id, SUM(sales) AS total_sales FROM orders GROUP BY customer_id
)
SELECT c.customer_name, ct.total_sales
FROM customer_totals ct JOIN customers c ON c.customer_id = ct.customer_id
ORDER BY ct.total_sales DESC LIMIT 5;
"""
pd.read_sql_query(query, conn)

,customer_name,total_sales
0,Sean Miller,25043.050
1,Tamara Chand,19052.218
2,Raymond Buch,15117.339
3,Tom Ashbrook,14595.620
4,Adrian Barton,14473.571


In [19]:
query = """
WITH customer_totals AS (
    SELECT customer_id, SUM(sales) AS total_sales FROM orders GROUP BY customer_id
)
SELECT c.customer_name, ct.total_sales
FROM customer_totals ct JOIN customers c ON c.customer_id = ct.customer_id
ORDER BY ct.total_sales ASC LIMIT 5;
"""
pd.read_sql_query(query, conn)

,customer_name,total_sales
0,Thais Sissman,4.833
1,Lela Donovan,5.304
2,Carl Jackson,16.520
3,Mitch Gastineau,16.739
4,Roy Skaria,22.328


In [20]:
query = """
SELECT o.customer_id, c.customer_name, COUNT(DISTINCT o.order_id) AS order_count
FROM orders o JOIN customers c ON c.customer_id = o.customer_id
GROUP BY o.customer_id, c.customer_name
HAVING COUNT(DISTINCT o.order_id) = 1;
"""
pd.read_sql_query(query, conn)

,customer_id,customer_name,order_count
0,AO-10810,Anthony O'Donnell,1
1,AR-10570,Anemone Ratner,1
2,CJ-11875,Carl Jackson,1
3,JC-15385,Jenna Caffey,1
4,JR-15700,Jocasta Rupert,1
5,LD-16855,Lela Donovan,1
6,MG-18205,Mitch Gastineau,1
7,PH-18790,Patricia Hirasaki,1
8,RE-19405,Ricardo Emerson,1
9,RM-19750,Roland Murray,1


In [21]:
query = """
WITH customer_totals AS (
    SELECT customer_id, SUM(sales) AS total_sales FROM orders GROUP BY customer_id
)
SELECT c.customer_name, ct.total_sales
FROM customer_totals ct JOIN customers c ON c.customer_id = ct.customer_id
WHERE ct.total_sales > (SELECT AVG(total_sales) FROM customer_totals)
ORDER BY ct.total_sales DESC;
"""
pd.read_sql_query(query, conn)

,customer_name,total_sales
0,Sean Miller,25043.050
1,Tamara Chand,19052.218
2,Raymond Buch,15117.339
3,Tom Ashbrook,14595.620
4,Adrian Barton,14473.571
...,...,...
289,Julie Kriz,2932.484
290,Shaun Weien,2921.544
291,Maris LaWare,2921.500
292,Rob Dowd,2912.894


In [22]:
query = """
SELECT c.customer_name, MAX(o.sales) AS highest_order_value
FROM orders o JOIN customers c ON c.customer_id = o.customer_id
GROUP BY c.customer_name
ORDER BY highest_order_value DESC;
"""
pd.read_sql_query(query, conn)

,customer_name,highest_order_value
0,Sean Miller,22638.480
1,Tamara Chand,17499.950
2,Raymond Buch,13999.960
3,Tom Ashbrook,11199.968
4,Hunter Lopez,10499.970
...,...,...
788,Carl Jackson,16.520
789,Mitch Gastineau,12.320
790,Roy Skaria,9.648
791,Lela Donovan,5.304


For all 793 customers, there is an extremely skewed distribution on the high end. The biggest selling customer is Sean Miller who has done total sales amounting to \$25,043.05 and is also the biggest seller of the highest selling individual item amounting to \$22,638.48 and is followed by Tamara Chand, Raymond Buch, Tom Ashbrook, and Adrian Barton. The bottom 5 customers have done nearly no spending (\$4.83 to \$22.33 in total), which amounts to a 5,000x difference from the biggest spender. There are 12 customers who have done only 1 transaction each and not returned after that. Average spending per customer comes to \$2,896.85 but only 294 out of 793 (~37%) are above this line in a right-skewed distribution where the average is pulled higher by a few bigger spenders, while the rest is doing less than the average. Conclusion: the spending has extremely right-skewed distribution and is dependent on the spending pattern of a smaller group of loyal customers who should be retained along with investigating the single-time buyers.
